# Análise completa de um caso

### Carregamento do caso

In [1]:
import pandas as pd
import diplo_grafo.config as cfg
import re
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

case = cases.iloc[0]

sentences = get_sentences(case)

print(case['case_text'])
print("\n==================================\n")
print("\n|| ".join([s.text for s in sentences]) + " ||\n")

2026-09-08 23:38:27.984 | INFO     | diplo_grafo.config:<module>:11 - PROJ_ROOT path is: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1


A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

In [2]:
for s in sentences:
    s.tokenize()
    print(s.tokens)
    

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right', 'flank', 'and', 'lower', 'quadrant', 'abdominal', 'pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast', 'enhanced', 'computed', 'tomography', ',', 'demonstrating', 'a', '6cm', 'cystic', 'lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA', ',', 'which', 'revealed', 'normal', 'pancreatic', 'echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal', 'septations', 'or', 'associated', 'masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA', 'of', 'the', 'cyst', 'demonstrated', 'no', 'evidence', 'of', 'malign

In [3]:
from diplo_grafo.clinical_terms import reconstruct_clinical_tokens

for sentence in sentences:
    sentence.tokenize()
    sentence.tokens = reconstruct_clinical_tokens(sentence.tokens)

for sentence in sentences:
    print(sentence.tokens)

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right flank', 'and', 'lower quadrant abdominal pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast enhanced computed tomography', ',', 'demonstrating', 'a', '6cm', 'cystic lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]', ',', 'which', 'revealed', 'normal', 'pancreatic echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal septations', 'or', 'associated masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA [fine needle aspiration]', 'of', 'the', 'cyst', '

### Construção do Grafo de Conhecimento com Atributos Ricos (Nodes & Edges)

A partir das sentenças com tokens reconstruídos obtidas na célula anterior, estruturamos o Grafo de Conhecimento clínico seguindo o esquema do diagrama Mermaid em `docs/steps.md`, com entidades e relações em inglês (conforme o visualizador do grafo [Diploghraph-Viewer](https://tochafh.github.io/Diploghraph-Viewer/)).

Integramos as expressões regulares desenvolvidas em `extracao_regex.ipynb` para enriquecimento semântico:
* **Atributos de Nós:** Propriedades intrínsecas da entidade (ex: idade e gênero do `Patient`, dimensões e valores laboratoriais quantificados do `Result`).
* **Atributos de Arestas:** Modulações da relação/ação clínica quando fizer sentido semântico:
  * `TAKES`: quantidade/dosagem da medicação (ex: `dosage=40 mg` ou `dosage=300 mg`).
  * `PRESENTS_WITH`: tempo de evolução do sintoma (ex: `duration=3-day history`).
  * `LOCATED_IN`: lateralidade (`laterality=right`) ou aspecto (`aspect=posterior`).
  * `DIAGNOSED_WITH`: status e método de confirmação (`status=confirmed; method=pathology` vs `status=suspected; method=clinical/imaging`).
  * `UNDERWENT_EXAM`: momento temporal do exame (ex: `timeline=POD 4`).

| Entidade no Mermaid (PT) | Tipo no Grafo (EN) | Prefixo ID | Atributos Típicos |
|---|---|---|---|
| Paciente | `Patient` | `P` | `age=...; gender=...` |
| Sintoma | `Symptom` | `S` | (intrínsecos ao sintoma) |
| Doença | `Disease` | `D` | (intrínsecos à condição) |
| Exame | `Exam` | `E` | (técnica/modalidade) |
| Resultado | `Result` | `R` | `size=...`, `value=...; unit=...`, `status=...` |
| PartesCorpo | `Anatomy` | `A` | (região/sistema) |
| Medicação | `Medication` | `M` | (princípio ativo) |
| Efeito | `Effect` | `Ef` | (efeito adverso) |

In [4]:
# ==============================================================================
# LÓGICA DE CONSTRUÇÃO DO GRAFO CLÍNICO COM ATRIBUTOS RICOS (NODES & EDGES)
# ==============================================================================
import re
import pandas as pd
import diplo_grafo.config as cfg

# 1. Expressões Regulares de Extração de Atributos (derivadas de extracao_regex.ipynb)
DIMENSION_3D = re.compile(
    r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)",
    re.IGNORECASE
)
DIMENSION_2D = re.compile(
    r"(\d+(?:\.\d+)?)\s*(cm|mm)\s*x\s*(\d+(?:\.\d+)?)\s*(cm|mm)",
    re.IGNORECASE
)
DIMENSION_1D = re.compile(
    r"\b(\d+(?:\.\d+)?)\s*(cm|mm)\b",
    re.IGNORECASE
)
LAB_MEASUREMENT = re.compile(
    r"(\d+(?:,\d+)*(?:\.\d+)?)\s*(mg/dL|g/dL|ng/mL|U/L|iu/ml|mm/h|mmol/L|g%|fL|/uL|x10\^9/L|K/uL|%)\b",
    re.IGNORECASE
)
DURATION_HISTORY = re.compile(
    r"\b(\d+)-(day|week|month|year)\s+history\b",
    re.IGNORECASE
)
DURATION_GENERIC = re.compile(
    r"\b(\d+-(?:day|week|month|year))\b",
    re.IGNORECASE
)
DOSAGE = re.compile(
    r"\b(\d+(?:\.\d+)?)\s*(mg|mcg|g|IU|units)\b",
    re.IGNORECASE
)
TIMELINE_POD = re.compile(
    r"\b(?:POD|postoperative day)\s*(\d+)\b",
    re.IGNORECASE
)
LATERALITY = re.compile(
    r"\b(right|left|bilateral)\b",
    re.IGNORECASE
)
ASPECT = re.compile(
    r"\b(posterior|anterior|superior|inferior|distal|proximal)\b",
    re.IGNORECASE
)

# Padrão para detecção de escopo de negação ou resolução clínica
NEGATION_PATTERN = re.compile(
    r"(?:no\s+evidence\s+of|negative\s+for|free\s+of|denies|denied|without|resolution\s+of|resolved|uninvolved)\s+([^.,;]+)",
    re.IGNORECASE
)

# 2. Vocabulário Clínico Estruturado (Alinhado ao Mermaid em steps.md)
ENTITY_CATEGORIES = {
    "Symptom": {
        "lower quadrant abdominal pain", "abdominal pain", "nausea", "constipation",
        "pain", "dyspnea", "fever", "chest pain", "shortness of breath", "headache"
    },
    "Disease": {
        "mucinous pancreatic cystic neoplasm", "mucinous pancreatic cyst",
        "gastric duplication cyst", "GDC [gastric duplication cyst]",
        "systemic lupus erythematosus", "ventricular septal defect",
        "malignancy", "adenocarcinoma", "heart failure", "tuberculosis"
    },
    "Exam": {
        "contrast enhanced computed tomography", "computed tomography",
        "EUS-FNA [endoscopic ultrasound-guided fine needle aspiration]",
        "FNA [fine needle aspiration]", "Intraoperative endoscopy",
        "oesophagogastroduodenoscopy", "Final pathology", "physical examination",
        "biopsy", "ultrasound"
    },
    "Result": {
        "cystic lesion", "pancreatic echotexture", "cyst",
        "extracellular mucin", "carcinoembryonic antigen",
        "carbohydrate antigen", "internal septations", "associated masses"
    },
    "Anatomy": {
        "right flank", "stomach", "pancreas", "posterior wall",
        "lesser sac", "coeliac axis", "coeliac vessels", "abdomen"
    },
    "Medication": {
        "antibiotics", "chemotherapy", "isoniazid", "rifampin",
        "ethambutol", "pyrazinamide", "aspirin", "omeprazole"
    }
}

TERM_TO_TYPE = {
    term.lower(): cat
    for cat, terms in ENTITY_CATEGORIES.items()
    for term in terms
}

def is_entity_negated(entity_str: str, text: str) -> bool:
    """Verifica se a entidade está sob escopo de negação ou resolução clínica."""
    # Divide a frase em orações adversativas para não propagar negações após conjunções
    clauses = re.split(r"\s+(?:but|however|although)\s+", text, flags=re.IGNORECASE)
    for clause in clauses:
        m = NEGATION_PATTERN.search(clause)
        if m and entity_str.lower() in m.group(1).lower():
            return True
    return False

def extract_knowledge_graph_rich(case_row, sentence_list):
    """
    Constrói os DataFrames de nós e arestas com distribuição semântica rica de atributos:
    - Atributos intrínsecos nos NÓS (Demografia, Medidas laboratoriais, Dimensões de achados).
    - Atributos relacionais nas ARESTAS (Duração de sintomas, Dosagem de fármacos, Lateralidade, Status de diagnóstico).
    """
    nodes = []
    edges = []
    node_ids = {}

    def get_or_create_node(label, node_type, attributes=""):
        key = (label.lower(), node_type)
        if key in node_ids:
            # Atualiza atributos se o novo registro trouxer detalhes adicionais
            for node in nodes:
                if node["node_id"] == node_ids[key]:
                    if not node["attributes"] and attributes:
                        node["attributes"] = attributes
                    elif attributes and attributes not in node["attributes"]:
                        node["attributes"] += f"; {attributes}"
            return node_ids[key]
        prefix = {
            "Patient": "P",
            "Symptom": "S",
            "Disease": "D",
            "Exam": "E",
            "Result": "R",
            "Anatomy": "A",
            "Medication": "M",
            "Effect": "Ef"
        }.get(node_type, "N")
        nid = f"{prefix}{len([k for k in node_ids if k[1] == node_type]) + 1}"
        node_ids[key] = nid
        nodes.append({
            "node_id": nid,
            "type": node_type,
            "label": label,
            "attributes": attributes
        })
        return nid

    def add_edge(source_id, target_id, relation, attributes=""):
        eid = f"e{len(edges) + 1}"
        edges.append({
            "edge_id": eid,
            "source_id": source_id,
            "target_id": target_id,
            "relation": relation,
            "attributes": attributes
        })
        return eid

    # 1. Nó do Paciente: atributos demográficos intrínsecos
    patient_attrs = f"age={case_row['age']}; gender={case_row['gender']}"
    patient_id = get_or_create_node(
        f"Patient {case_row['case_id']}",
        "Patient",
        patient_attrs
    )

    # 2. Processamento sentencial com contextualização de atributos
    for s in sentence_list:
        txt = s.text.lower()
        orig_text = s.text

        # Ignora sentenças irrelevantes ou de histórico não-contributório
        if "non-contributory" in txt or "were normal" in txt:
            continue

        # Identifica entidades candidatas presentes nos tokens
        found_entities = []
        for tok in s.tokens:
            tok_lower = tok.lower()
            if tok_lower in TERM_TO_TYPE:
                found_entities.append((tok, TERM_TO_TYPE[tok_lower]))

        # Filtra entidades negadas ou resolvidas
        active_entities = [e for e in found_entities if not is_entity_negated(e[0], orig_text)]

        symptoms = [e[0] for e in active_entities if e[1] == "Symptom"]
        diseases = [e[0] for e in active_entities if e[1] == "Disease"]
        exams = [e[0] for e in active_entities if e[1] == "Exam"]
        results = [e[0] for e in active_entities if e[1] == "Result"]
        anatomy = [e[0] for e in active_entities if e[1] == "Anatomy"]
        meds = [e[0] for e in active_entities if e[1] == "Medication"]

        # Extração de Dimensões (priorizando 3D > 2D > 1D)
        sizes = []
        spans_sz = set()
        for m in DIMENSION_3D.finditer(orig_text):
            sizes.append((m.start(), m.end(), m.group(0).strip()))
            spans_sz.update(range(m.start(), m.end()))
        for m in DIMENSION_2D.finditer(orig_text):
            if not any(i in spans_sz for i in range(m.start(), m.end())):
                sizes.append((m.start(), m.end(), m.group(0).strip()))
                spans_sz.update(range(m.start(), m.end()))
        for m in DIMENSION_1D.finditer(orig_text):
            if not any(i in spans_sz for i in range(m.start(), m.end())):
                sizes.append((m.start(), m.end(), m.group(0).strip()))

        # Mapeamento de posições dos resultados para atribuição de medidas
        res_spans = []
        for r in results:
            m = re.search(re.escape(r), orig_text, re.IGNORECASE)
            if m:
                res_spans.append((m.start(), m.end(), r))

        # Atribuição precisa de dimensões à entidade Result mais próxima
        assigned_sizes = {}
        for s_start, s_end, sz in sizes:
            if res_spans:
                def dist_sz(ent):
                    start, end, _ = ent
                    if end <= s_start: return s_start - end
                    if start >= s_end: return start - s_end
                    return 0
                closest = min(res_spans, key=dist_sz)
                if dist_sz(closest) < 30:
                    assigned_sizes[closest[2]] = sz

        # Extração e associação de medições laboratoriais
        labs = [(m.start(), m.end(), m.group(1), m.group(2)) for m in LAB_MEASUREMENT.finditer(orig_text)]
        assigned_labs = {r: [] for r in results}
        for l_start, l_end, val, unit in labs:
            preceding = [ent for ent in res_spans if ent[1] <= l_start]
            if preceding:
                closest = min(preceding, key=lambda e: l_start - e[1])
                if (l_start - closest[1]) < 60:
                    assigned_labs[closest[2]].append(f"value={val}; unit={unit}")

        # Dosagens
        dosages = [(m.start(), m.end(), m.group(0).strip()) for m in DOSAGE.finditer(orig_text)]

        # Duração / Linha do tempo
        dur_match = DURATION_HISTORY.search(orig_text) or DURATION_GENERIC.search(orig_text)
        duration_val = dur_match.group(0) if dur_match else ""

        pod_match = TIMELINE_POD.search(orig_text)
        timeline_val = pod_match.group(0) if pod_match else ""

        lat_match = LATERALITY.search(orig_text)
        laterality_val = lat_match.group(0).lower() if lat_match else ""

        asp_match = ASPECT.search(orig_text)
        aspect_val = asp_match.group(0).lower() if asp_match else ""

        # --- APLICAÇÃO DAS REGRAS SEMÂNTICAS DO GRAFO ---

        # 1. Relação: Patient --PRESENTS_WITH--> Symptom
        # Atributo de tempo colocado na ARESTA
        if symptoms and any(w in txt for w in ["presented", "history", "pain", "complained", "had", "symptom"]):
            pres_attr = f"duration={duration_val}" if duration_val else ""
            for sym in symptoms:
                sid = get_or_create_node(sym, "Symptom")
                add_edge(patient_id, sid, "PRESENTS_WITH", pres_attr)
                
                # Se for dor e houver anatomia, vincula com lateralidade na aresta
                if "pain" in sym.lower():
                    for anat in anatomy:
                        loc_attr = f"laterality={laterality_val}" if laterality_val else ""
                        aid = get_or_create_node(anat, "Anatomy")
                        add_edge(sid, aid, "LOCATED_IN", loc_attr)

        # 2. Relação: Patient --TAKES--> Medication
        # Atributo de quantidade/dosagem colocado na ARESTA
        if meds:
            for med in meds:
                mid = get_or_create_node(med, "Medication")
                m_match = re.search(re.escape(med), orig_text, re.IGNORECASE)
                dose_attr = ""
                if m_match and dosages:
                    closest_dose = sorted(dosages, key=lambda d: abs(d[0] - m_match.end()))[0][2]
                    dose_attr = f"dosage={closest_dose}"
                elif dosages:
                    dose_attr = f"dosage={dosages[0][2]}"
                add_edge(patient_id, mid, "TAKES", dose_attr)

        # 3. Relação: Patient --UNDERWENT_EXAM--> Exam
        # Atributo de linha do tempo colocado na ARESTA
        if exams:
            for ex in exams:
                ex_edge_attr = f"timeline={timeline_val}" if timeline_val else ""
                eid = get_or_create_node(ex, "Exam")
                add_edge(patient_id, eid, "UNDERWENT_EXAM", ex_edge_attr)

                # Exam --REVEALS--> Result
                # Atributos de medida/quantidade colocados no NÓ DE RESULTADO
                for res in results:
                    res_attrs = []
                    if res in assigned_sizes:
                        res_attrs.append(f"size={assigned_sizes[res]}")
                    if assigned_labs.get(res):
                        res_attrs.extend(assigned_labs[res])
                    if "normal" in txt and "echotexture" in res.lower():
                        res_attrs.append("status=normal")

                    rid = get_or_create_node(res, "Result", "; ".join(res_attrs))
                    add_edge(eid, rid, "REVEALS")

                    for anat in anatomy:
                        loc_attr = f"aspect={aspect_val}" if aspect_val else ""
                        aid = get_or_create_node(anat, "Anatomy")
                        add_edge(rid, aid, "LOCATED_IN", loc_attr)

        # 4. Relação: Patient --DIAGNOSED_WITH / HAS_HISTORY--> Disease
        # Atributos de status e método colocados na ARESTA
        if diseases:
            for dis in diseases:
                did = get_or_create_node(dis, "Disease")
                if "history" in txt and ("medical" in txt or "past" in txt):
                    rel = "HAS_FAMILY_HISTORY" if "family" in txt else "HAS_PERSONAL_HISTORY"
                    edge_attr = ""
                else:
                    rel = "DIAGNOSED_WITH"
                    diag_attrs = []
                    if "pathology" in txt or "final pathology" in txt:
                        diag_attrs.append("status=confirmed")
                        diag_attrs.append("method=pathology")
                    elif "suggesting" in txt or "suspected" in txt:
                        diag_attrs.append("status=suspected")
                        diag_attrs.append("method=clinical/imaging")
                    else:
                        diag_attrs.append("status=confirmed")
                    edge_attr = "; ".join(diag_attrs)

                add_edge(patient_id, did, rel, edge_attr)

                for anat in anatomy:
                    loc_attr = f"aspect={aspect_val}" if aspect_val else ""
                    aid = get_or_create_node(anat, "Anatomy")
                    add_edge(did, aid, "LOCATED_IN", loc_attr)

    return pd.DataFrame(nodes), pd.DataFrame(edges)

nodes_df, edges_df = extract_knowledge_graph_rich(case, sentences)

# Salvar as tabelas para visualização
proc_dir = cfg.PROCESSED_DATA_DIR
proc_dir.mkdir(parents=True, exist_ok=True)
nodes_df.to_csv(proc_dir / "nodes.csv", index=False)
edges_df.to_csv(proc_dir / "edges.csv", index=False)

print("Grafo com atributos ricos construído com sucesso!")
print(f"Total de Nós: {len(nodes_df)}")
print(f"Total de Arestas: {len(edges_df)}")
print(f"Arquivos exportados para: {proc_dir / 'nodes.csv'} e {proc_dir / 'edges.csv'}")


Grafo com atributos ricos construído com sucesso!
Total de Nós: 22
Total de Arestas: 25
Arquivos exportados para: /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1/data/processed/nodes.csv e /home/raphael/Desktop/faculdade/MC896_NLP/Diplotokens-P1/project 1/data/processed/edges.csv


In [5]:
print("=== TABELA DE NÓS (nodes.csv) ===")
display(nodes_df.head(20))

print("\n=== TABELA DE ARESTAS COM ATRIBUTOS (edges.csv) ===")
display(edges_df[edges_df["attributes"] != ""])

print("\n=== AMOSTRA GERAL DE ARESTAS ===")
display(edges_df.head(15))


=== TABELA DE NÓS (nodes.csv) ===


,node_id,type,label,attributes
0,P1,Patient,Patient PMC5137649_01,age=44.0; gender=Female
1,S1,Symptom,lower quadrant abdominal pain,
2,A1,Anatomy,right flank,
3,S2,Symptom,nausea,
4,S3,Symptom,constipation,
5,E1,Exam,contrast enhanced computed tomography,
6,R1,Result,cystic lesion,size=6cm
7,A2,Anatomy,stomach,
8,A3,Anatomy,pancreas,
9,E2,Exam,EUS-FNA [endoscopic ultrasound-guided fine nee...,



=== TABELA DE ARESTAS COM ATRIBUTOS (edges.csv) ===


,edge_id,source_id,target_id,relation,attributes
0,e1,P1,S1,PRESENTS_WITH,duration=3-day history
1,e2,S1,A1,LOCATED_IN,laterality=right
2,e3,P1,S2,PRESENTS_WITH,duration=3-day history
3,e4,P1,S3,PRESENTS_WITH,duration=3-day history
18,e19,P1,D1,DIAGNOSED_WITH,status=suspected; method=clinical/imaging
20,e21,P1,D2,DIAGNOSED_WITH,status=confirmed
21,e22,D2,A2,LOCATED_IN,aspect=posterior
22,e23,D2,A4,LOCATED_IN,aspect=posterior
24,e25,P1,D3,DIAGNOSED_WITH,status=confirmed; method=pathology



=== AMOSTRA GERAL DE ARESTAS ===


,edge_id,source_id,target_id,relation,attributes
0,e1,P1,S1,PRESENTS_WITH,duration=3-day history
1,e2,S1,A1,LOCATED_IN,laterality=right
2,e3,P1,S2,PRESENTS_WITH,duration=3-day history
3,e4,P1,S3,PRESENTS_WITH,duration=3-day history
4,e5,P1,E1,UNDERWENT_EXAM,
5,e6,E1,R1,REVEALS,
6,e7,R1,A2,LOCATED_IN,
7,e8,R1,A3,LOCATED_IN,
8,e9,P1,E2,UNDERWENT_EXAM,
9,e10,E2,R2,REVEALS,


In [6]:
from IPython.display import display, Markdown

def generate_mermaid_code(nodes, edges):
    lines = ["```mermaid", "graph LR"]
    for _, r in nodes.iterrows():
        nid = r["node_id"]
        lbl = str(r["label"]).replace('"', "'")
        ntype = r["type"]
        attrs = str(r["attributes"]).replace('"', "'") if pd.notna(r.get("attributes")) and r.get("attributes") else ""
        attr_str = f"<br/><small>{attrs}</small>" if attrs else ""
        
        if ntype == "Patient":
            lines.append(f'    {nid}(("{lbl}{attr_str}"))')
        else:
            lines.append(f'    {nid}["{lbl} <br/> <i>({ntype})</i>{attr_str}"]')

    for _, r in edges.iterrows():
        src = r["source_id"]
        tgt = r["target_id"]
        rel = r["relation"]
        attrs = str(r["attributes"]).replace('"', "'") if pd.notna(r.get("attributes")) and r.get("attributes") else ""
        rel_label = f"{rel}<br/><i>[{attrs}]</i>" if attrs else rel
        lines.append(f'    {src} -- "{rel_label}" --> {tgt}')
    lines.append("```")
    return "\n".join(lines)

mermaid_md = generate_mermaid_code(nodes_df, edges_df)
display(Markdown(mermaid_md))
print(mermaid_md)


```mermaid
graph LR
    P1(("Patient PMC5137649_01<br/><small>age=44.0; gender=Female</small>"))
    S1["lower quadrant abdominal pain <br/> <i>(Symptom)</i>"]
    A1["right flank <br/> <i>(Anatomy)</i>"]
    S2["nausea <br/> <i>(Symptom)</i>"]
    S3["constipation <br/> <i>(Symptom)</i>"]
    E1["contrast enhanced computed tomography <br/> <i>(Exam)</i>"]
    R1["cystic lesion <br/> <i>(Result)</i><br/><small>size=6cm</small>"]
    A2["stomach <br/> <i>(Anatomy)</i>"]
    A3["pancreas <br/> <i>(Anatomy)</i>"]
    E2["EUS-FNA [endoscopic ultrasound-guided fine needle aspiration] <br/> <i>(Exam)</i>"]
    R2["pancreatic echotexture <br/> <i>(Result)</i><br/><small>status=normal</small>"]
    R3["cyst <br/> <i>(Result)</i><br/><small>size=6cm x 9cm</small>"]
    E3["FNA [fine needle aspiration] <br/> <i>(Exam)</i>"]
    R4["extracellular mucin <br/> <i>(Result)</i>"]
    R5["carcinoembryonic antigen <br/> <i>(Result)</i><br/><small>value=12,476.5; unit=ng/ml</small>"]
    R6["carbohydrate antigen <br/> <i>(Result)</i><br/><small>value=6; unit=iu/ml</small>"]
    D1["mucinous pancreatic cystic neoplasm <br/> <i>(Disease)</i>"]
    E4["Intraoperative endoscopy <br/> <i>(Exam)</i>"]
    D2["gastric duplication cyst <br/> <i>(Disease)</i>"]
    A4["posterior wall <br/> <i>(Anatomy)</i>"]
    E5["oesophagogastroduodenoscopy <br/> <i>(Exam)</i>"]
    D3["GDC [gastric duplication cyst] <br/> <i>(Disease)</i>"]
    P1 -- "PRESENTS_WITH<br/><i>[duration=3-day history]</i>" --> S1
    S1 -- "LOCATED_IN<br/><i>[laterality=right]</i>" --> A1
    P1 -- "PRESENTS_WITH<br/><i>[duration=3-day history]</i>" --> S2
    P1 -- "PRESENTS_WITH<br/><i>[duration=3-day history]</i>" --> S3
    P1 -- "UNDERWENT_EXAM" --> E1
    E1 -- "REVEALS" --> R1
    R1 -- "LOCATED_IN" --> A2
    R1 -- "LOCATED_IN" --> A3
    P1 -- "UNDERWENT_EXAM" --> E2
    E2 -- "REVEALS" --> R2
    R2 -- "LOCATED_IN" --> A2
    E2 -- "REVEALS" --> R3
    R3 -- "LOCATED_IN" --> A2
    P1 -- "UNDERWENT_EXAM" --> E3
    E3 -- "REVEALS" --> R3
    E3 -- "REVEALS" --> R4
    E3 -- "REVEALS" --> R5
    E3 -- "REVEALS" --> R6
    P1 -- "DIAGNOSED_WITH<br/><i>[status=suspected; method=clinical/imaging]</i>" --> D1
    P1 -- "UNDERWENT_EXAM" --> E4
    P1 -- "DIAGNOSED_WITH<br/><i>[status=confirmed]</i>" --> D2
    D2 -- "LOCATED_IN<br/><i>[aspect=posterior]</i>" --> A2
    D2 -- "LOCATED_IN<br/><i>[aspect=posterior]</i>" --> A4
    P1 -- "UNDERWENT_EXAM" --> E5
    P1 -- "DIAGNOSED_WITH<br/><i>[status=confirmed; method=pathology]</i>" --> D3
```

```mermaid
graph LR
    P1(("Patient PMC5137649_01<br/><small>age=44.0; gender=Female</small>"))
    S1["lower quadrant abdominal pain <br/> <i>(Symptom)</i>"]
    A1["right flank <br/> <i>(Anatomy)</i>"]
    S2["nausea <br/> <i>(Symptom)</i>"]
    S3["constipation <br/> <i>(Symptom)</i>"]
    E1["contrast enhanced computed tomography <br/> <i>(Exam)</i>"]
    R1["cystic lesion <br/> <i>(Result)</i><br/><small>size=6cm</small>"]
    A2["stomach <br/> <i>(Anatomy)</i>"]
    A3["pancreas <br/> <i>(Anatomy)</i>"]
    E2["EUS-FNA [endoscopic ultrasound-guided fine needle aspiration] <br/> <i>(Exam)</i>"]
    R2["pancreatic echotexture <br/> <i>(Result)</i><br/><small>status=normal</small>"]
    R3["cyst <br/> <i>(Result)</i><br/><small>size=6cm x 9cm</small>"]
    E3["FNA [fine needle aspiration] <br/> <i>(Exam)</i>"]
    R4["extracellular mucin <br/> <i>(Result)</i>"]
    R5["carcinoembryonic antigen <br/> <i>(Result)</i><br/><small>value=12,476.5; unit=ng/ml</small>"]
    R6["carbohydrat